In [55]:
# Core data manipulation
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical tools
from scipy import stats

## About the Dataset

**Dataset:** OWID COVID-19 Dataset  
**Source:** https://github.com/owid  
**File:** `owid-covid-data.csv`   

In [16]:
owid = pd.read_csv('owid-covid-data.csv')
display(owid.head())

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [35]:
# shape
print(owid.shape)
print(f'\nThere are {owid.shape[0]} rows and {owid.shape[1]} columns\n')
print('- Each row is one case of COVID')

(429435, 67)

There are 429435 rows and 67 columns

- Each row is one case of COVID


In [45]:
# 2. & 3 Column Names & Data Types
print("\nColumn Names and Data Types:")
for col in owid.columns:
    print(f"  - {col}: {owid[col].dtype}")


Column Names and Data Types:
  - iso_code: object
  - continent: object
  - location: object
  - date: object
  - total_cases: float64
  - new_cases: float64
  - new_cases_smoothed: float64
  - total_deaths: float64
  - new_deaths: float64
  - new_deaths_smoothed: float64
  - total_cases_per_million: float64
  - new_cases_per_million: float64
  - new_cases_smoothed_per_million: float64
  - total_deaths_per_million: float64
  - new_deaths_per_million: float64
  - new_deaths_smoothed_per_million: float64
  - reproduction_rate: float64
  - icu_patients: float64
  - icu_patients_per_million: float64
  - hosp_patients: float64
  - hosp_patients_per_million: float64
  - weekly_icu_admissions: float64
  - weekly_icu_admissions_per_million: float64
  - weekly_hosp_admissions: float64
  - weekly_hosp_admissions_per_million: float64
  - total_tests: float64
  - new_tests: float64
  - total_tests_per_thousand: float64
  - new_tests_per_thousand: float64
  - new_tests_smoothed: float64
  - new_te

The dataset contains 67 variables including identifiers (e.g., country codes),
temporal variables (date), and epidemiological measures such as cases,
deaths, and vaccinations.

In [48]:
classification = []

for col in owid.columns:
    if col in {'date', 'iso_code'}:
        classification.append({
            'column': col,
            'data_type': 'identifier / temporal',
            'measurement_type': 'not applicable'
        })
    elif owid[col].dtype == 'object':
        classification.append({
            'column': col,
            'data_type': 'categorical',
            'measurement_type': 'categorical (nominal)'
        })
    else:
        values = owid[col].dropna()
        if len(values) > 0 and ((values % 1) == 0).all():
            measurement = 'discrete'
        else:
            measurement = 'continuous'

        classification.append({
            'column': col,
            'data_type': 'quantitative',
            'measurement_type': measurement
        })

classification_df = pd.DataFrame(classification)
classification_df

,column,data_type,measurement_type
0,iso_code,identifier / temporal,not applicable
1,continent,categorical,categorical (nominal)
2,location,categorical,categorical (nominal)
3,date,identifier / temporal,not applicable
4,total_cases,quantitative,discrete
...,...,...,...
62,population,quantitative,discrete
63,excess_mortality_cumulative_absolute,quantitative,continuous
64,excess_mortality_cumulative,quantitative,continuous
65,excess_mortality,quantitative,continuous


Numeric variables representing counts (e.g., cases, deaths) are treated as
**discrete**, while rates and averages are treated as **continuous**.
Identifier and date columns are excluded from quantitative analysis.

In [50]:
# categorical columns
categorical_cols = owid.select_dtypes(include='object').columns

categorical_values = {
    col: owid[col].dropna().unique()
    for col in categorical_cols
}

categorical_values

{'iso_code': array(['AFG', 'OWID_AFR', 'ALB', 'DZA', 'ASM', 'AND', 'AGO', 'AIA', 'ATG',
        'ARG', 'ARM', 'ABW', 'OWID_ASI', 'AUS', 'AUT', 'AZE', 'BHS', 'BHR',
        'BGD', 'BRB', 'BLR', 'BEL', 'BLZ', 'BEN', 'BMU', 'BTN', 'BOL',
        'BES', 'BIH', 'BWA', 'BRA', 'VGB', 'BRN', 'BGR', 'BFA', 'BDI',
        'KHM', 'CMR', 'CAN', 'CPV', 'CYM', 'CAF', 'TCD', 'CHL', 'CHN',
        'COL', 'COM', 'COG', 'COK', 'CRI', 'CIV', 'HRV', 'CUB', 'CUW',
        'CYP', 'CZE', 'COD', 'DNK', 'DJI', 'DMA', 'DOM', 'TLS', 'ECU',
        'EGY', 'SLV', 'OWID_ENG', 'GNQ', 'ERI', 'EST', 'SWZ', 'ETH',
        'OWID_EUR', 'OWID_EUN', 'FRO', 'FLK', 'FJI', 'FIN', 'FRA', 'GUF',
        'PYF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GIB', 'GRC', 'GRL',
        'GRD', 'GLP', 'GUM', 'GTM', 'GGY', 'GIN', 'GNB', 'GUY', 'HTI',
        'OWID_HIC', 'HND', 'HKG', 'HUN', 'ISL', 'IND', 'IDN', 'IRN', 'IRQ',
        'IRL', 'IMN', 'ISR', 'ITA', 'JAM', 'JPN', 'JEY', 'JOR', 'KAZ',
        'KEN', 'KIR', 'OWID_KOS', 'KWT', 'KGZ', '

In [51]:
categorical_summary = []

for col in categorical_cols:
    counts = owid[col].value_counts()
    categorical_summary.append({
        'column': col,
        'unique_values': owid[col].nunique(),
        'mode': counts.idxmax(),
        'mode_count': counts.max()
    })

pd.DataFrame(categorical_summary)

,column,unique_values,mode,mode_count
0,iso_code,255,OWID_HIC,3026
1,continent,6,Africa,95419
2,location,255,High-income countries,3026
3,date,1688,2022-01-10,261
4,tests_units,4,tests performed,80099


For categorical variables, the mode was identified as the most frequently
occurring category based on value counts.

In [49]:
# numeric columns
numeric_cols = owid.select_dtypes(include='number').columns

quant_summary = []

for col in numeric_cols:
    quant_summary.append({
        'column': col,
        'min': owid[col].min(),
        'max': owid[col].max(),
        'mean': owid[col].mean(),
        'median': owid[col].median(),
        'std': owid[col].std()
    })

quant_df = pd.DataFrame(quant_summary)
quant_df

,column,min,max,mean,median,std
0,total_cases,0.00,7.758668e+08,7.365292e+06,63653.00,4.477582e+07
1,new_cases,0.00,4.423623e+07,8.017360e+03,0.00,2.296649e+05
2,new_cases_smoothed,0.00,6.319461e+06,8.041026e+03,12.00,8.661611e+04
3,total_deaths,0.00,7.057132e+06,8.125957e+04,799.00,4.411901e+05
4,new_deaths,0.00,1.037190e+05,7.185214e+01,0.00,1.368323e+03
...,...,...,...,...,...,...
57,population,47.00,7.975105e+09,1.520336e+08,6336393.00,6.975408e+08
58,excess_mortality_cumulative_absolute,-37726.10,1.349776e+06,5.604765e+04,6815.20,1.568691e+05
59,excess_mortality_cumulative,-44.23,7.808000e+01,9.766431e+00,8.13,1.204066e+01
60,excess_mortality,-95.92,3.782200e+02,1.092535e+01,5.66,2.456071e+01


Statistics are reported only for variables where they are meaningful.
Count-based variables (e.g., cases, deaths, vaccinations) are measured in
**number of people**, while rate-based variables are measured in **percentages
or per-capita units**.

Identifier variables (e.g., codes or IDs) are excluded, as summary statistics
would be misleading.

In [54]:
skip_cols = {'iso_code', 'date'}

summary = []

for col in owid.columns:
    missing_pct = owid[col].isna().mean() * 100

    # Categorical
    if owid[col].dtype == 'object':
        summary.append({
            'Column': col,
            'Type': 'Categorical',
            'Measurement': 'Nominal',
            'Missing %': round(missing_pct, 2),
            'Details': f"Unique values: {owid[col].nunique()}, "
                       f"Mode: {owid[col].mode().iloc[0]}"
        })

    # Numeric
    else:
        values = owid[col].dropna()
        measurement = (
            'Discrete'
            if len(values) > 0 and ((values % 1) == 0).all()
            else 'Continuous'
        )

        summary.append({
            'Column': col,
            'Type': 'Quantitative',
            'Measurement': measurement,
            'Missing %': round(missing_pct, 2),
            'Details': (
                f"Range: {values.min():.2f}–{values.max():.2f}, "
                f"Mean: {values.mean():.2f}, "
                f"Median: {values.median():.2f}, "
                f"Std: {values.std():.2f}"
            )
        })

summary_df = pd.DataFrame(summary)

summary_df = summary_df.sort_values(by=['Type', 'Measurement', 'Missing %'], ascending=[True, True, False])

summary_df

,Column,Type,Measurement,Missing %,Details
33,tests_units,Categorical,Nominal,75.13,"Unique values: 4, Mode: tests performed"
1,continent,Categorical,Nominal,6.18,"Unique values: 6, Mode: Africa"
0,iso_code,Categorical,Nominal,0.00,"Unique values: 255, Mode: OWID_HIC"
2,location,Categorical,Nominal,0.00,"Unique values: 255, Mode: High-income countries"
3,date,Categorical,Nominal,0.00,"Unique values: 1688, Mode: 2021-02-15"
...,...,...,...,...,...
5,new_cases,Quantitative,Discrete,4.49,"Range: 0.00–44236227.00, Mean: 8017.36, Median..."
8,new_deaths,Quantitative,Discrete,4.38,"Range: 0.00–103719.00, Mean: 71.85, Median: 0...."
4,total_cases,Quantitative,Discrete,4.11,"Range: 0.00–775866783.00, Mean: 7365292.35, Me..."
7,total_deaths,Quantitative,Discrete,4.11,"Range: 0.00–7057132.00, Mean: 81259.57, Median..."


The table above summarizes all variables in the dataset, including data type,
measurement scale, missingness, and relevant descriptive statistics.
Statistics are reported only where meaningful.